# 로컬 VoD 데이터로 모델 추론 (노트북)

웹 없이 **가져온 View-of-Delft(KITTI 레이아웃)** 데이터로 다음을 실행합니다.

1. **레이더** — `sklearn.DBSCAN` 기하 클러스터 → 탐지 후보 (학습 가중치 없음, `ai-inference`의 `/infer/vod/radar-fusion`과 동일 로직)
2. **카메라** — **Ultralytics YOLOv8** 객체 검출 (가중치 경로는 아래 셀에서 지정)
3. **LiDAR** (선택) — 동일 프레임 `velodyne` `.bin`이 있으면, 1위 레이더 클러스터 중심 주변 반경에서 점 수로 **간단 검증**

## 준비

- Jupyter/VS Code에서 이 파일을 연 뒤 **작업 디렉터리(cwd)를 `vod-devkit`** 으로 두는 것을 권장합니다 (`cd vod-devkit` 후 `jupyter lab`).
- 데이터 루트: 환경변수 `VOD_ROOT` 또는 아래 셀의 `ROOT_DIR`.
- **프레임 자동 선택**: `VOD_FRAME_MODE=random`(기본) / `first` / `fixed` + `VOD_FRAME_ID=00000`. `VOD_RANDOM_SEED`로 재현 가능.
- 의존성: `numpy`, `matplotlib`, `scikit-learn`, `opencv-python`, `ultralytics`, `torch` (YOLO용).

In [ ]:
# 최초 1회만 실행해도 됩니다.
# %pip install -q numpy matplotlib scikit-learn opencv-python ultralytics torch

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN

NOTEBOOK_ROOT = Path.cwd().resolve()
if not (NOTEBOOK_ROOT / "vod").is_dir():
    raise RuntimeError(
        f"cwd에 vod 패키지가 없습니다: {NOTEBOOK_ROOT}\n"
        "터미널에서 cd vod-devkit 후 노트북을 다시 여세요."
    )
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

REPO_ROOT = NOTEBOOK_ROOT.parent
AI_INF = REPO_ROOT / "ai-inference"

# ========= 사용자 설정 =========
_default_data = NOTEBOOK_ROOT / "vod-received" / "view_of_delft_PUBLIC"
ROOT_DIR = os.environ.get("VOD_ROOT", str(_default_data if _default_data.is_dir() else NOTEBOOK_ROOT / "example_set"))
# 프레임: random(동기 풀에서 무작위) | first | fixed(아래 FRAME_ID_FIXED)
FRAME_MODE = os.environ.get("VOD_FRAME_MODE", "random").strip().lower()
FRAME_ID_FIXED = os.environ.get("VOD_FRAME_ID", "00000")
_rs = os.environ.get("VOD_RANDOM_SEED")
RANDOM_SEED = int(_rs) if _rs is not None and str(_rs).strip() != "" else None
YOLO_WEIGHTS = os.environ.get("YOLO_MODEL", str(AI_INF / "yolov8n.pt"))
DBSCAN_EPS = float(os.environ.get("VOD_RADAR_DBSCAN_EPS", "4.0"))
DBSCAN_MIN_SAMPLES = int(os.environ.get("VOD_RADAR_DBSCAN_MIN_SAMPLES", "3"))
LIDAR_VALIDATE_RADIUS_M = 2.5
# ==============================

print("ROOT_DIR:", ROOT_DIR)
print("FRAME_MODE:", FRAME_MODE, "| fixed:", FRAME_ID_FIXED, "| RANDOM_SEED:", RANDOM_SEED)
print("YOLO_WEIGHTS:", YOLO_WEIGHTS, "exists:", Path(YOLO_WEIGHTS).is_file())
print("DBSCAN eps / min_samples:", DBSCAN_EPS, DBSCAN_MIN_SAMPLES)

## 1) 프레임 로드 (VoD devkit)

`FrameDataLoader`로 레이더·이미지를 읽습니다. LiDAR는 파일이 있을 때만 메모리에 올립니다.

In [ ]:
from vod.configuration import KittiLocations
from vod.frame import FrameDataLoader

kitti = KittiLocations(root_dir=ROOT_DIR, output_dir=str(NOTEBOOK_ROOT / "_nb_model_out"), frame_set_path="", pred_dir="")


def list_synced_frame_stems(loc):
    cam = Path(loc.camera_dir)
    rad = Path(loc.radar_dir)
    jpgs = {p.stem for p in cam.glob("*.jpg")}
    bins = {p.stem for p in rad.glob("*.bin")}
    common = jpgs & bins

    def stem_key(s):
        try:
            return int(s, 10)
        except ValueError:
            return s

    return sorted(common, key=stem_key)


_synced = list_synced_frame_stems(kitti)
if not _synced:
    raise RuntimeError("image_2(.jpg)와 radar velodyne(.bin)의 stem이 겹치는 프레임이 없습니다.")
_mode = (FRAME_MODE or "random").strip().lower()
if _mode == "fixed":
    resolved_frame_id = FRAME_ID_FIXED
elif _mode == "first":
    resolved_frame_id = _synced[0]
else:
    rng = np.random.default_rng(RANDOM_SEED)
    resolved_frame_id = str(rng.choice(np.array(_synced)))
print(
    f"프레임 선택: mode={_mode!r} → {resolved_frame_id!r} (동기 풀 {len(_synced)}개, 예: {_synced[:3]}…)"
)
FRAME_ID = resolved_frame_id

loader = FrameDataLoader(kitti_locations=kitti, frame_number=FRAME_ID)

rad = loader.radar_data
img_rgb = loader.image
lid = loader.lidar_data
lidar_xyz = None if lid is None else lid[:, :3]

if lidar_xyz is not None:
    print("LiDAR points:", lidar_xyz.shape[0], "|", Path(kitti.lidar_dir) / f"{loader.file_id}.bin")
else:
    print("LiDAR 없음 (선택)")

if rad is None:
    raise RuntimeError("레이더 .bin 을 읽지 못했습니다. FRAME_ID와 radar/training/velodyne 경로를 확인하세요.")
if img_rgb is None:
    raise RuntimeError("이미지 .jpg 를 읽지 못했습니다. lidar/training/image_2 경로와 파일명(프레임 번호)을 확인하세요.")

print("레이더 shape (N,7):", rad.shape)
print("이미지 shape:", img_rgb.shape)

## 1b) 대시보드형 시각화 (실제 레이더 포인트)

웹 프로토타입과 같은 구성: **Range–Azimuth**, **지면 XY + 도플러 색**, **3D 부분 샘플**. (전체 점이 많으면 3D만 서브샘플)

In [ ]:
xyz = rad[:, :3]
v_comp = rad[:, 5]
rng_pt = np.linalg.norm(xyz, axis=1)
az_pt = np.degrees(np.arctan2(xyz[:, 1], xyz[:, 0]))
n3d = min(8000, xyz.shape[0])
idx3d = np.random.default_rng(RANDOM_SEED).choice(xyz.shape[0], size=n3d, replace=False)

fig = plt.figure(figsize=(14, 5))
ax0 = fig.add_subplot(1, 3, 1)
sc0 = ax0.scatter(az_pt, rng_pt, s=4, c=v_comp, cmap="coolwarm", alpha=0.65)
ax0.set_xlabel("azimuth (deg)")
ax0.set_ylabel("range (m)")
ax0.set_title("Range vs azimuth (color=Doppler)")
plt.colorbar(sc0, ax=ax0, label="v_r comp (m/s)")

ax1 = fig.add_subplot(1, 3, 2)
sc1 = ax1.scatter(xyz[:, 0], xyz[:, 1], s=4, c=v_comp, cmap="viridis", alpha=0.65)
ax1.set_xlabel("x east (m)")
ax1.set_ylabel("y north (m)")
ax1.set_title("Ground x–y (color=Doppler)")
ax1.set_aspect("equal")
plt.colorbar(sc1, ax=ax1, label="v_r comp (m/s)")

ax2 = fig.add_subplot(1, 3, 3, projection="3d")
ax2.scatter(xyz[idx3d, 0], xyz[idx3d, 1], xyz[idx3d, 2], s=2, c=v_comp[idx3d], cmap="plasma", alpha=0.5)
ax2.set_xlabel("x")
ax2.set_ylabel("y")
ax2.set_zlabel("z")
ax2.set_title(f"3D 샘플 ({n3d} pts)")

plt.suptitle(f"frame {FRAME_ID} · raw radar points")
plt.tight_layout()
plt.show()

## 2) 레이더 — DBSCAN 클러스터 (`ai-inference`와 동일)

In [ ]:
def radar_clusters_dbscan(pts: np.ndarray, eps: float, min_samples: int) -> list[dict]:
    xyz = pts[:, :3]
    rcs = pts[:, 3]
    v_comp = pts[:, 5]
    if xyz.shape[0] == 0:
        return []
    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(xyz)
    labels = clustering.labels_
    out: list[dict] = []
    for lab in sorted(set(labels.tolist())):
        if lab < 0:
            continue
        m = labels == lab
        c = xyz[m].mean(axis=0)
        rng = float(np.linalg.norm(c))
        azimuth_deg = float(np.degrees(np.arctan2(c[1], c[0])))
        elevation_deg = float(
            np.degrees(np.arctan2(c[2], np.sqrt(c[0] ** 2 + c[1] ** 2) + 1e-6))
        )
        vd = v_comp[m]
        doppler_mps = float(np.mean(vd)) if vd.size else 0.0
        rc = rcs[m]
        rcs_mean = float(np.mean(rc)) if rc.size else 0.0
        npts = int(m.sum())
        conf = min(
            0.99,
            0.25
            + 0.02 * min(npts, 20)
            + 0.15 * min(abs(doppler_mps) / 8.0, 1.0)
            + 0.1 * min(rcs_mean / 30.0, 1.0),
        )
        out.append(
            {
                "id": f"cluster-{lab}",
                "rangeM": round(rng, 2),
                "azimuthDeg": round(azimuth_deg, 2),
                "elevationDeg": round(elevation_deg, 2),
                "dopplerMps": round(doppler_mps, 3),
                "confidence": round(conf, 3),
                "clusterSize": npts,
                "centroidM": [round(float(c[0]), 3), round(float(c[1]), 3), round(float(c[2]), 3)],
            }
        )
    out.sort(key=lambda d: d["confidence"], reverse=True)
    return out[:12]


radar_dets = radar_clusters_dbscan(rad, DBSCAN_EPS, DBSCAN_MIN_SAMPLES)
print(f"레이더 클러스터 후보: {len(radar_dets)}개 (상위 최대 12)")
for d in radar_dets[:5]:
    print(d)

## 3) 카메라 — YOLOv8

가중치가 없으면 Ultralytics가 자동으로 내려받습니다. 커스텀 `.pt` 경로를 `YOLO_WEIGHTS`에 두면 됩니다.

In [ ]:
from ultralytics import YOLO

model = YOLO(YOLO_WEIGHTS)
# BGR for OpenCV-style; loader.image는 RGB라고 가정 → YOLO는 RGB도 받음
results = model.predict(source=img_rgb, verbose=False)
r0 = results[0]
plot_bgr = r0.plot()  # BGR uint8
plot_rgb = plot_bgr[..., ::-1]

fig, ax = plt.subplots(1, 1, figsize=(14, 5))
ax.imshow(plot_rgb)
ax.set_title(f"YOLO | {Path(YOLO_WEIGHTS).name} | frame {FRAME_ID}")
ax.axis("off")
plt.tight_layout()
plt.show()

if r0.boxes is None or len(r0.boxes) == 0:
    print("검출 없음")
else:
    names = r0.names
    for box in r0.boxes[:15]:
        ci = int(box.cls[0].item())
        conf = float(box.conf[0].item())
        print(names.get(ci, str(ci)), f"conf={conf:.3f}")

## 4) LiDAR 검증 (선택)

레이더·LiDAR를 **같은 차량/라이다 좌표계**로 본다는 가정입니다. 실제 오차가 크면 캘리브 기반 정합이 필요합니다.

In [ ]:
def _bearing_deg_xy(x: float, y: float) -> float:
    return float(np.degrees(np.arctan2(y, x)))


def _angle_diff_abs_deg(a: float, b: float) -> float:
    d = (a - b + 180.0) % 360.0 - 180.0
    return abs(d)


def lidar_validate_cluster(
    lidar_xyz: np.ndarray,
    centroid: list[float],
    radius_m: float,
    *,
    radar_range_m: float,
    radar_azimuth_deg: float,
) -> dict:
    """ai-inference main.py 와 동일한 요약 지표."""
    c = np.array(centroid, dtype=np.float64)
    d = np.linalg.norm(lidar_xyz - c, axis=1)
    inside = d < radius_m
    n = int(inside.sum())
    if n == 0:
        return {
            "matched": False,
            "pointsInRoi": 0,
            "meanDistanceM": None,
            "lidarClusterRangeM": None,
            "deltaRangeM": None,
            "deltaBearingDeg": None,
            "iouBevProxy": 0.0,
            "verdict": "불일치",
        }
    lid_roi = lidar_xyz[inside]
    lid_cent = lid_roi.mean(axis=0)
    lid_range = float(np.linalg.norm(lid_cent))
    lid_az = _bearing_deg_xy(float(lid_cent[0]), float(lid_cent[1]))
    rr = float(radar_range_m)
    ra = float(radar_azimuth_deg)
    delta_r = round(abs(rr - lid_range), 3)
    delta_bear = round(_angle_diff_abs_deg(ra, lid_az), 3)
    iou_proxy = min(0.99, 0.35 + min(n, 200) / 200.0 * 0.55) if n >= 5 else min(0.5, 0.15 + n * 0.02)
    matched = n >= 5
    verdict = "일치" if matched and delta_r < 15.0 and delta_bear < 5.0 else ("부분" if matched else "불일치")
    return {
        "matched": matched,
        "pointsInRoi": n,
        "meanDistanceM": round(float(d[inside].mean()), 3),
        "lidarClusterRangeM": round(lid_range, 3),
        "radarRangeM": round(rr, 3),
        "deltaRangeM": delta_r,
        "deltaBearingDeg": delta_bear,
        "lidarClusterAzimuthDeg": round(lid_az, 3),
        "iouBevProxy": round(float(iou_proxy), 3),
        "verdict": verdict,
    }


if lidar_xyz is None or not radar_dets:
    print("스킵: LiDAR 없음 또는 레이더 후보 없음")
else:
    primary = radar_dets[0]
    lv = lidar_validate_cluster(
        lidar_xyz,
        primary["centroidM"],
        LIDAR_VALIDATE_RADIUS_M,
        radar_range_m=float(primary["rangeM"]),
        radar_azimuth_deg=float(primary["azimuthDeg"]),
    )
    print("1위 클러스터:", primary["id"], primary["centroidM"])
    print(f"LiDAR 전체 점 {lidar_xyz.shape[0]}개, 반경 {LIDAR_VALIDATE_RADIUS_M}m:", lv)

## 5) (참고) 레이더 BEV

클러스터 색으로 상위 후보만 강조합니다.

### 4b) LiDAR BEV (실제 XY, 1위 레이더 ROI)

상위 뷰에서 레이더 1위 클러스터 중심 주변 점만 강조합니다.

In [ ]:
if lidar_xyz is None or not radar_dets:
    print("BEV 스킵")
else:
    primary = radar_dets[0]
    c = np.array(primary["centroidM"], dtype=np.float64)
    d = np.linalg.norm(lidar_xyz - c, axis=1)
    in_roi = d < LIDAR_VALIDATE_RADIUS_M
    lid_bev = lidar_xyz[in_roi]
    # 너무 많으면 표시만 서브샘플
    if lid_bev.shape[0] > 12_000:
        lid_bev = lid_bev[:: max(1, lid_bev.shape[0] // 12_000)]

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.set_facecolor("#0f172a")
    fig.patch.set_facecolor("#1e293b")
    ax.scatter(lidar_xyz[:: max(1, lidar_xyz.shape[0] // 25_000), 0], lidar_xyz[:: max(1, lidar_xyz.shape[0] // 25_000), 1], s=1, c="#475569", alpha=0.35)
    if len(lid_bev):
        ax.scatter(lid_bev[:, 0], lid_bev[:, 1], s=4, c="#34d399", alpha=0.85, label="LiDAR ROI")
    ax.scatter([0], [0], s=120, c="#facc15", marker="^", edgecolors="#854d0e", zorder=5, label="ego")
    ax.scatter([c[0]], [c[1]], s=80, c="none", edgecolors="#a78bfa", linewidths=2, zorder=6, label="레이더 1위 중심(xy)")
    ax.plot([0, c[0]], [0, c[1]], "m--", alpha=0.6, linewidth=1)
    ax.set_aspect("equal")
    ax.set_xlabel("x (m)", color="#e2e8f0")
    ax.set_ylabel("y (m)", color="#e2e8f0")
    ax.tick_params(colors="#94a3b8")
    ax.set_title(f"LiDAR BEV · frame {FRAME_ID}", color="#f1f5f9")
    ax.legend(loc="upper right", fontsize=8, facecolor="#334155", labelcolor="#e2e8f0")
    for spine in ax.spines.values():
        spine.set_color("#475569")
    plt.tight_layout()
    plt.show()

In [ ]:
xyz = rad[:, :3]
labels = np.full(xyz.shape[0], -1, dtype=np.int32)
clust = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES).fit_predict(xyz)
fig, ax = plt.subplots(figsize=(7, 7))
noise = clust < 0
ax.scatter(xyz[noise, 0], xyz[noise, 1], s=2, c="lightgray", alpha=0.25, label="noise")
for lab in sorted(set(clust.tolist())):
    if lab < 0:
        continue
    m = clust == lab
    ax.scatter(xyz[m, 0], xyz[m, 1], s=6, alpha=0.7, label=f"c{lab}")
ax.set_aspect("equal")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("레이더 XY (DBSCAN)")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right", fontsize=7, ncol=2)
plt.tight_layout()
plt.show()